<a href="https://colab.research.google.com/github/Seixin/Basic-Sentiment-Analysis/blob/main/SentimenAnalisisVidYT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install Sastrawi

In [28]:
import pandas as pd
from googleapiclient.discovery import build
from google.colab import files
from google.colab import drive
from google.colab import userdata
import re
import string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [29]:
API_KEY = userdata.get('SECRET')

video_ids = [
    'oM9v6P41tRA',
    'G1TRm2B-h1M',
    '9-poYwCZxDQ'
]

youtube = build('youtube', 'v3', developerKey=API_KEY)
semua_komentar=[]

for video_id in video_ids:
  print(f'Sedang menarik data dari video_id:{video_id}...')

  request = youtube.commentThreads().list(
      part='snippet',
      videoId = video_id,
      maxResults= 100,
      textFormat = 'plainText'
  )

  jumlah_diambil=0
  batas_maksimal=200

  while request is not None and jumlah_diambil < batas_maksimal:
    response = request.execute()
    for item in response['items']:
      komentar=item['snippet']['topLevelComment']['snippet']['textDisplay']
      author=item['snippet']['topLevelComment']['snippet']['authorDisplayName']

      semua_komentar.append({
          'Video_ID':video_id,
          'Author':author,
          'Komentar':komentar
      })

      jumlah_diambil+=1
      if jumlah_diambil >= batas_maksimal:
        break

    if 'nextPageToken' in response and jumlah_diambil < batas_maksimal:
      request = youtube.commentThreads().list_next(
          previous_request=request,
          previous_response=response
      )
    else:
      break

df= pd.DataFrame(semua_komentar)
nama_file="dataset_gadgetin.csv"
df.to_csv(nama_file, index=False, encoding='utf-8')

print(f"\nSelesai! Total komentar ditarik: {len(df)}")
print("Mendownload file otomatis...")

files.download(nama_file)

Sedang menarik data dari video_id:oM9v6P41tRA...
Sedang menarik data dari video_id:G1TRm2B-h1M...
Sedang menarik data dari video_id:9-poYwCZxDQ...

Selesai! Total komentar ditarik: 600
Mendownload file otomatis...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
drive.mount('/content/drive')

path='/content/drive/MyDrive/SentimenAnalisis/dataset_gadgetin.xlsx'
df = pd.read_excel(path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Video_ID,Author,Komentar,sentiment
0,oM9v6P41tRA,@robbyexplore,Poco x8 pro max ini untuk video konten kamera ...,1
1,oM9v6P41tRA,@Generate_music-7,Tahan dulu mana tau ada lagi PX9 😅,0
2,oM9v6P41tRA,@BaniHamdun,Bang @David...review hp kentang kamera bening ...,0
3,oM9v6P41tRA,@tinanovita-ym5pi,infinix gt 50 pro,0
4,oM9v6P41tRA,@irfanwiyadi4359,banggg request nantiii kalaw main game nya.. m...,0


In [31]:
stop_factory = StopWordRemoverFactory()
stopword = stop_factory.create_stop_word_remover()
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def clean_text(text):
  text = text.lower()
  text = re.sub(r'http\S+|wwww\S+|https\S+', '', text, flags=re.MULTILINE)
  text = re.sub(r'@\w+|#\w+', '', text)
  text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
  text = stopword.remove(text)
  text = stemmer.stem(text)
  return text

df['clean_comment'] = df['Komentar'].apply(clean_text)
df[['clean_comment', 'Komentar']].head()

,clean_comment,Komentar
0,poco x pro max untuk video konten kamera bagus,Poco x8 pro max ini untuk video konten kamera ...
1,tahan dulu mana tau lagi px,Tahan dulu mana tau ada lagi PX9 😅
2,bang review hp kentang kamera bening bening em...,Bang @David...review hp kentang kamera bening ...
3,infinix gt pro,infinix gt 50 pro
4,banggg request nantiii kalaw main game nya mai...,banggg request nantiii kalaw main game nya.. m...


In [32]:
df = df.dropna(subset=["clean_comment"])
df = df[df['clean_comment']!= '']

In [33]:
print(f'jumlah duplikat sebelum dihapus {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'jumlah duplikat setelah dihapus {df.duplicated().sum()}')


jumlah duplikat sebelum dihapus 9
jumlah duplikat setelah dihapus 0


In [34]:
df['sentiment'].value_counts()

,count
sentiment,
0,295
2,172
1,121


In [35]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_comment'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Total fitur/kata unik (Vocabulary):", X.shape[1])
print("Jumlah data latih (X_train):", X_train.shape[0])
print("Jumlah data uji (X_test):", X_test.shape[0])

Total fitur/kata unik (Vocabulary): 1894
Jumlah data latih (X_train): 470
Jumlah data uji (X_test): 118
